# UTS Data Science - Dokumentasi Praktikum
## Pertemuan 3: Data Cleaning (Missing Values, Outlier & Ekstraksi Data)

* **Nama Lengkap:** Martin Hotmatua Siregar
* **NIM:** 240401020111
* **Kelas:** IF403
* **Program Studi:** PJJ Informatika
* **Instansi:** Universitas Siber Asia

## Bagian 1: Pipeline Data Cleaning (Dataset Perumahan)
Di bagian ini, kita akan melakukan simulasi pembersihan data pada data properti perumahan yang memiliki beberapa masalah kualitas data (M1: Missing values, M2: Duplikat, M3: Outlier, M4: Inkonsistensi format teks). Karena file `housing_dirty.csv` merupakan data simulasi praktikum, kita akan membuat replika datanya langsung di dalam kode agar pipeline pembersihan data dapat berjalan otomatis tanpa kendala eksternal.

In [1]:
# Import library dasar yang dibutuhkan untuk manipulasi dan pembersihan data
import pandas as pd
import numpy as np

# STEP 0: Membuat data simulasi sesuai dengan spesifikasi dataset housing_dirty.csv di modul
# Dataset ini memiliki 7 kolom dan sengaja dikotori dengan duplikat, missing, dan outlier
data_kotor = {
    'id': [1, 2, 3, 4, 5, 5, 6, 7, 8, 9],
    'luas_m2': [120, np.nan, 85, 200, 150, 150, 450, 90, np.nan, 110], # ada missing & outlier 450
    'harga_juta': [1200, 850, np.nan, 3500, 1800, 1800, 9500, 900, 1100, np.nan], # ada missing & outlier 9500
    'kota': ['jakarta ', ' Bandung', 'jakarta', 'Surabaya', 'medan', 'medan', 'Jakarta', 'Bandung', ' surabaya', 'Jakarta'], # spasi & huruf kapital berantakan
    'kamar': [3, 2, 2, 4, np.nan, np.nan, 6, 2, 3, 2], # ada missing value
    'tahun_bangun': [2015, 2010, 2018, 1945, 2020, 2020, 2012, 1930, 2017, 2014], # ada tahun di bawah 1950 (outlier)
    'kondisi': ['Baik', 'baik', 'Perlu Renovasi', 'baik', 'Bagus', 'Bagus', 'baik', 'perlu renovasi', 'Bagus', 'Baik'] # spasi berantakan
}

df = pd.DataFrame(data_kotor)
print("=== SHAPE AWAL DATASET ===")
print(df.shape)
print("\n=== JUMLAH MISSING VALUES PER KOLOM ===")
print(df.isnull().sum())

# STEP 1: Menghapus baris data yang duplikat total
df.drop_duplicates(inplace=True)
print(f"\nSetelah proses penghapusan duplikat, ukuran data menjadi: {df.shape}")

# STEP 2: Normalisasi string/teks untuk mengatasi inkonsistensi format penulisan
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# STEP 3: Imputasi nilai yang hilang (Missing Values)
# Menggunakan nilai Median untuk kolom numerik, dan Modus untuk kolom kategorik
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# STEP 4: Menangani nilai ekstrem (Outlier) menggunakan batas IQR Fence dengan fungsi clip()
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    # Membatasi nilai ekstrem agar masuk kembali ke dalam pagar batas IQR (Capping)
    df[col] = df[col].clip(batas_bawah, batas_atas)

# STEP 5: Validasi akhir untuk menjamin kualitas data sebelum diekspor
assert df.duplicated().sum() == 0, 'Peringatan: Masih ada data duplikat!'
assert df.isnull().sum().sum() == 0, 'Peringatan: Masih ada data kosong!'

print("\n=== SHAPE AKHIR DATASET BERSIH ===")
print(df.shape)

# Menampilkan hasil dataset yang sudah bersih secara keseluruhan
df

=== SHAPE AWAL DATASET ===
(10, 7)

=== JUMLAH MISSING VALUES PER KOLOM ===
id              0
luas_m2         2
harga_juta      2
kota            0
kamar           2
tahun_bangun    0
kondisi         0
dtype: int64

Setelah proses penghapusan duplikat, ukuran data menjadi: (9, 7)

=== SHAPE AKHIR DATASET BERSIH ===
(9, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,120.0,1200.0,Jakarta,3.0,2015.0,baik
1,2,120.0,850.0,Bandung,2.0,2010.0,baik
2,3,85.0,1200.0,Jakarta,2.0,2018.0,perlu renovasi
3,4,200.0,2850.0,Surabaya,4.0,1999.5,baik
4,5,150.0,1800.0,Medan,2.0,2020.0,bagus
6,6,210.0,2850.0,Jakarta,6.0,2012.0,baik
7,7,90.0,900.0,Bandung,2.0,1999.5,perlu renovasi
8,8,120.0,1100.0,Surabaya,3.0,2017.0,bagus
9,9,110.0,1200.0,Jakarta,2.0,2014.0,baik


**Analisis Temuan Bagian 1 (Pipeline Cleaning):**
Melalui simulasi pengerjaan pipeline di atas, saya berhasil mendeteksi dan menyelesaikan empat masalah utama kualitas data. Baris data yang duplikat duplikasi ID berhasil dipangkas, teks pada kolom kota yang awalnya berantakan spasi dan kapitalnya sukses diseragamkan menjadi format penulisan judul (*Title Case*), dan baris kosong pada data numerik berhasil ditambal secara aman menggunakan nilai tengah (median) kelompok. Terakhir, nilai properti ekstrem yang menyimpang jauh (seperti tarif rumah bernilai fantastis atau tahun pembuatan di bawah 1950) berhasil dijinakkan dengan metode pembatasan nilai (*Capping*) menggunakan batas pagar IQR Fence.

## Bagian 2: Ekstraksi Data dari REST API Publik via JSON
Pada tahap ini, kita mempraktikkan proses pengambilan data semi-terstruktur dari internet menggunakan REST API publik penyedia data latihan, yaitu JSONPlaceholder. Data diambil menggunakan metode HTTP GET, divalidasi kode statusnya, kemudian dikonversi menjadi bentuk tabel terstruktur menggunakan fungsi standardisasi Pandas.

In [2]:
# Mengimpor library requests untuk melakukan koneksi jaringan dan HTTP request
import requests
from pandas import json_normalize

# Menentukan alamat endpoint REST API publik yang dituju
url_api = "https://jsonplaceholder.typicode.com/users"

# Mengirimkan permintaan pengambilan data (GET Request) dengan batasan waktu tunggu (timeout) 10 detik
response = requests.get(url_api, timeout=10)

# Melakukan pengecekan kode status respons HTTP (200 berarti koneksi berhasil dan sukses)
if response.status_code == 200:
    data_json = response.json()

    # Mengonversi data JSON terikat/bersarang (nested JSON) menjadi DataFrame yang rata (flat)
    df_users = json_normalize(data_json, sep='_')

    # Menampilkan 5 kolom utama yang paling krusial dari hasil ekstraksi data pengguna
    print("Ekstraksi data pengguna dari API sukses dilakukan!")
    display(df_users[['id', 'name', 'email', 'address_city', 'company_name']].head())
else:
    print(f"Proses ekstraksi data gagal. Kode kesalahan server: {response.status_code}")

Ekstraksi data pengguna dari API sukses dilakukan!


,id,name,email,address_city,company_name
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC


**Analisis Temuan Bagian 2 (Ekstraksi API):**
Pengujian akses jaringan ke server REST API luar berjalan dengan sukses dan mengembalikan kode status HTTP `200` (OK). Objek JSON yang dikirimkan oleh server memiliki struktur data bersarang (*nested JSON*), di mana informasi alamat (*address*) dan data perusahaan (*company*) tersimpan di dalam struktur objek utama. Dengan memanfaatkan modul fungsi `json_normalize()` dari Pandas, struktur bersarang tersebut berhasil diurai secara otomatis menjadi kolom-kolom baru yang terpisah (`address_city` dan `company_name`) sehingga datanya menjadi siap untuk dianalisis lebih lanjut.

## Kesimpulan & Refleksi Pembelajaran (Sesi 3)

### 1. Apa yang Dipelajari?
Di pertemuan ketiga ini, saya belajar bahwa kualitas data adalah pondasi utama sebelum melangkah ke proses pemodelan atau machine learning (prinsip *Garbage In, Garbage Out*). Saya mempraktikkan cara mendeteksi data rusak, mengoreksi teks yang tidak konsisten, menambal data kosong dengan nilai statistik median, serta meredam efek data pencilan (*outlier*) menggunakan batas pagar IQR Fence. Selain pembersihan, saya juga belajar cara menembak data langsung dari internet menggunakan HTTP request ke REST API publik.

### 2. Temuan Utama
* Aktivitas pembersihan dan persiapan data (*data preparation*) memakan porsi waktu kerja terbesar seorang praktisi data science, yaitu sekitar 37% dari total waktu kerja harian.
* Penggunaan fungsi pembatasan nilai (*capping/clipping*) pada metode IQR Fence terbukti lebih aman dan bijak karena kita tidak perlu membuang baris data penting secara utuh, melainkan hanya menormalkan nilai yang terlampau ekstrem agar masuk ke rentang sebaran data yang wajar.
* Mengambil data dari web API membutuhkan penanganan ekstra seperti penentuan *timeout* dan pengecekan kode status HTTP untuk mencegah kegagalan sistem akibat kendala koneksi atau server yang mati.

### 3. Keterbatasan & Pertanyaan yang Muncul
* **Keterbatasan:** Proses pengisian data kosong pada praktikum ini masih menggunakan nilai statistik tunggal (median). Cara ini berisiko mengurangi variasi alami dari data jika jumlah data yang hilang terlampau banyak.
* **Pertanyaan:** Bagaimana cara yang tepat untuk melakukan pembersihan data jika jenis datanya bukan berupa angka atau tabel, melainkan teks panjang berantakan seperti data komentar atau cuitan di media sosial? Dan kapan waktu yang ideal untuk memilih menghapus total data pencilan dibanding membatasi nilainya dengan *capping*?